In [1]:
import xarray as xr
import numpy as np
import torch
import torch.nn as nn
from sklearn.model_selection import KFold
from torch.utils.data import DataLoader
import os
import pandas as pd

from src.dataset import LazyWeatherDataset
from src.preprocessing import flatten_target_dataset, standardize_with_stats, compute_overall_from_daily_stats
from src.models import get_model, get_model_input_dims
from src.train_loop import evaluate 

In [2]:
def evaluate_model_shash(
    X, y, stats, model_name,
    n_splits=5,
    batch_size=64,
    optimizer_class=torch.optim.Adam,
    lr=1e-3,
    criterion=None,
    level=None,
    latest=True,
):

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    days = X.day.values
    kf = KFold(n_splits=n_splits, shuffle=False)

    overall_stats = compute_overall_from_daily_stats(stats)

    all_nlls = []
    val_counts = []

    for fold, (train_idx, val_idx) in enumerate(kf.split(days)):

        val_counts.append(val_idx.shape[0])

        model_spec = (
            f"{model_name}/level={level}/"
            f"opt={optimizer_class.__name__}_lr={lr}_batch={batch_size}_"
            f"crit=SHASHLoss/fold={fold}"
        )

        model_dict = f"models/{model_spec}/"

        train_days = days[train_idx]
        val_days = days[val_idx]

        X_train = X.sel(day=train_days)
        X_val = X.sel(day=val_days)

        y_train = y.sel(time=train_days)
        y_val = y.sel(time=val_days)

        fold_stats = compute_overall_from_daily_stats(stats.sel(day=train_days))

        conversion_stats = xr.Dataset({
            v: ((fold_stats[v] - overall_stats[v]) / overall_stats[v.replace('_mean', '_std')])
            if v.endswith('_mean')
            else (fold_stats[v] / overall_stats[v])
            for v in fold_stats.data_vars
        })

        X_train_standardized = standardize_with_stats(X_train, conversion_stats)
        X_val_standardized = standardize_with_stats(X_val, conversion_stats)

        input_dimensions = get_model_input_dims(model_name)

        train_ds = LazyWeatherDataset(
            X_train_standardized,
            y=flatten_target_dataset(y_train),
            input_dimensions=input_dimensions
        )

        val_ds = LazyWeatherDataset(
            X_val_standardized,
            y=flatten_target_dataset(y_val),
            input_dimensions=input_dimensions
        )

        train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
        val_loader = DataLoader(val_ds, batch_size=batch_size)

        x_example, y_example = next(iter(train_loader))
        input_dim = x_example.shape[1:] if x_example.ndim > 2 else x_example.shape[1]
        output_dim = y_example.shape[1]

        model = get_model(model_name, input_dim, output_dim, t=None).to(device)

        if latest:
            weight_path = os.path.join(model_dict, "latest.pt")
        else:
            weight_path = os.path.join(model_dict, "best.pt")

        model.load_state_dict(
            torch.load(weight_path, map_location=torch.device("cpu"))["model_state_dict"]
        )

        model.eval()

        # -------------------------------------------------------
        # Collect per-sample per-target NLL
        # -------------------------------------------------------

        fold_nlls = []

        with torch.no_grad():
            for xb, yb in val_loader:
                xb = xb.to(device)
                yb = yb.to(device)

                pred_params = model(xb)

                # Expect shape: [batch, n_targets]
                nll = criterion(pred_params, yb, reduction="none")

                fold_nlls.append(nll.cpu().numpy())

        fold_nlls = np.concatenate(fold_nlls, axis=0)
        all_nlls.append(fold_nlls)

    all_nlls = np.concatenate(all_nlls, axis=0)

    mean_nll_per_target = all_nlls.mean(axis=0)

    return val_counts, mean_nll_per_target, model_spec.split('/fold')[0]

In [ ]:
latest = False

if latest:
    latest_str = 'latest_'
else:
    latest_str = 'best_'

input_dir = "/glade/work/milesep/convective_outlook_ml"
target_dir = "data/processed_data"
stats_dir = "data/processed_data"

nll_path = "results/" + latest_str + "all_shash_nll.csv"
rmse_path = "results/" + latest_str + "all_shash_rmse_mean.csv"
rmse_units_path = "results/" + latest_str + "all_shash_rmse_units_mean.csv"

# model_names = ["cnn3d_3_layer", "cnn3d_dropout_5_0", "cnn3d_dropout_5_0", "cnn3d_dropout_5_5", "cnn3d_dropout_5_5"]
# levels = ["small", "small", "small", "small", "small"]
# lrs = [1e-3, 1e-3, 1e-3, 1e-3, 1e-3]
# batch_sizes = [8, 32, 64, 32, 64]

# for name, level, lr, batch_size in zip(model_names, levels, lrs, batch_sizes):

results_df = pd.read_csv('results/results.csv')
model_names = results_df['model']

if os.path.exists(nll_path):
    nll_df = pd.read_csv(nll_path)
    done_model_specs = nll_df['model']
else:
    done_model_specs = []
id = 0
for model in model_names:
    # print(model)
    id += 1
    if "ShashNLL" not in model:
        continue
    if 'batch' in model and not (model.split('/')[0] in ['predict_mean', 'predict_zero']) and not (model in list(done_model_specs)):
        name = model.split('/')[0]
        level = model.split('level=')[1].split('/')[0]
        lr = float(model.split('lr=')[1].split('_')[0])
        batch_size = int(model.split('batch=')[1].split('_')[0])
        if level[:4] == 'slgt':
            slgt_mod_str = '_slgt'
        else:
            slgt_mod_str = ''
        print(id, name, level, lr, batch_size)
        if True:
            # print("LEVEL!!!!!" + level)
            lev = level.removesuffix("_new")
            inputs = xr.open_zarr(f"{input_dir}/train_inputs_{lev}.zarr")
            if level.endswith("_new"):
                tars = xr.open_dataset(f"{target_dir}/train_targets{slgt_mod_str}_new.nc")
            else:
                tars = xr.open_dataset(f"{target_dir}/train_targets{slgt_mod_str}.nc")
            stats = xr.open_dataset(f"{stats_dir}/daily_input_stats_{lev}.nc")
            sizes, losses, predictions, targets, model_spec = evaluate_model_shash(inputs, tars, stats, name, batch_size = batch_size, lr = lr, level = level, latest = latest)
    
            all_preds = np.array(predictions[0])
            all_targets = np.array(targets[0])

            for i in range(1, len(predictions)):
                all_preds = np.append(all_preds, np.array(predictions[i]), axis=0)
                all_targets = np.append(all_targets, np.array(targets[i]), axis=0)

            # --- SHASH PARAM UNPACK ---
            N, D = all_preds.shape
            K = all_targets.shape[1] if all_targets.ndim > 1 else 1

            params = all_preds.reshape(N, K, 4)

            mu    = params[..., 0]
            sigma = np.exp(params[..., 1])
            gamma = params[..., 2]
            tau   = np.exp(params[..., 3])

            y = all_targets.reshape(N, K)

            # --- COMPUTE NLL ---
            from scipy.stats import norm

            z = (y - mu) / sigma

            asinh_z = np.arcsinh(z)
            sinh_term = np.sinh(tau * asinh_z - gamma)

            log_pdf = (
                np.log(tau)
                - np.log(sigma)
                - 0.5 * np.log(1 + z**2)
                - 0.5 * (sinh_term ** 2)
            )

            nll = -log_pdf
            mean_nll = nll.mean(axis=0)

            # --- OPTIONAL: RMSE of mean (diagnostic only) ---
            mses = ((mu - y) ** 2).mean(axis=0)
            rmses = np.sqrt(mses)

            stds = tars['train_std'].values.flatten()
            rmse_units = rmses * stds

            # --- SAVE ---
            nll_row = {
                "model": model_spec,
                **{str(i): v for i, v in enumerate(mean_nll)}
            }

            rmse_row = {
                "model": model_spec,
                **{str(i): v for i, v in enumerate(rmses)}
            }

            rmse_units_row = {
                "model": model_spec,
                **{str(i): v for i, v in enumerate(rmse_units)}
            }
    
            os.makedirs(os.path.dirname(nll_path), exist_ok=True)
            os.makedirs(os.path.dirname(rmse_path), exist_ok=True)
            os.makedirs(os.path.dirname(rmse_units_path), exist_ok=True)
    
            if os.path.exists(nll_path):
                df = pd.read_csv(nll_path)
                df = df[df["model"] != nll_row["model"]]  # overwrite if it exists
                df = pd.concat([df, pd.DataFrame([nll_row])], ignore_index=True)
            else:
                df = pd.DataFrame([nll_row])
    
            df.to_csv(nll_path, index=False)
    
            if os.path.exists(rmse_path):
                df = pd.read_csv(rmse_path)
                df = df[df["model"] != rmse_row["model"]]  # overwrite if it exists
                df = pd.concat([df, pd.DataFrame([rmse_row])], ignore_index=True)
            else:
                df = pd.DataFrame([rmse_row])
    
            df.to_csv(rmse_path, index=False)
    
            if os.path.exists(rmse_units_path):
                df = pd.read_csv(rmse_units_path)
                df = df[df["model"] != rmse_units_row["model"]]  # overwrite if it exists
                df = pd.concat([df, pd.DataFrame([rmse_units_row])], ignore_index=True)
            else:
                df = pd.DataFrame([rmse_units_row])
    
            df.to_csv(rmse_units_path, index=False)